# 🎬 SadTalker — Lip Sync pour Réseaux Sociaux

Génère une vidéo animée avec synchronisation des lèvres à partir d'une **image PNG** et d'un **fichier audio MP3/WAV**.
Export automatique au format **vertical 9:16** (TikTok, Instagram Reels, YouTube Shorts).

---

| Étape | Description |
|---|---|
| 🔧 **1** | Vérification GPU + installation des dépendances |
| 📦 **2** | Clonage de SadTalker + téléchargement des modèles |
| 🖼️ **3** | Upload de l'image source (PNG) |
| 🎵 **4** | Upload de l'audio (MP3 / WAV) |
| 🚀 **5** | Génération de la vidéo |
| 📱 **6** | Export 9:16 + téléchargement |

> **Prérequis :** Activer le GPU T4 dans *Exécution → Modifier le type d'exécution → GPU*

---
## 🔧 Étape 1 — Vérification du GPU et installation

In [ ]:
# ─── Vérification du GPU ───────────────────────────────────────────────────────
# SadTalker nécessite obligatoirement un GPU pour une génération rapide.
# Si 'No GPU' apparaît → Exécution > Modifier le type d'exécution > GPU T4

import subprocess, sys

def run(cmd):
    """Exécute une commande shell et affiche la sortie."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0 and result.stderr:
        print("[ERREUR]", result.stderr[-2000:])
    return result.returncode == 0

print("=" * 60)
print("  VÉRIFICATION DU MATÉRIEL")
print("=" * 60)

try:
    import torch
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ GPU détecté : {name}")
        print(f"   VRAM        : {vram:.1f} GB")
        print(f"   CUDA        : {torch.version.cuda}")
    else:
        print("❌ Aucun GPU détecté ! Activez le GPU avant de continuer.")
        sys.exit()
except ImportError:
    print("⚠️  PyTorch non encore importé — normal au premier lancement.")

run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
print("=" * 60)

In [ ]:
# ─── Installation des dépendances système ─────────────────────────────────────
# ffmpeg  : conversion et formatage vidéo (obligatoire pour l'export 9:16)
# git-lfs : téléchargement des gros fichiers de modèles depuis Hugging Face

print("📦 Installation des outils système...")
run("apt-get update -qq && apt-get install -y -qq ffmpeg git-lfs libgl1")
run("git lfs install")
print("✅ Outils système installés.")

---
## 📦 Étape 2 — Clonage de SadTalker et téléchargement des modèles

In [ ]:
# ─── Clonage du dépôt SadTalker ───────────────────────────────────────────────
# On se place dans /content pour que tout soit dans le répertoire de travail Colab.
# Le flag --depth 1 télécharge uniquement le dernier commit (plus rapide).

import os

SADTALKER_DIR = "/content/SadTalker"
os.chdir("/content")

if not os.path.isdir(SADTALKER_DIR):
    print("🔽 Clonage de SadTalker...")
    ok = run("git clone --depth 1 https://github.com/OpenTalker/SadTalker.git")
    if ok:
        print("✅ SadTalker cloné avec succès.")
    else:
        print("❌ Erreur lors du clonage — vérifiez votre connexion internet.")
else:
    print("✅ SadTalker déjà présent, on passe.")

os.chdir(SADTALKER_DIR)

In [ ]:
# ─── Correctifs de compatibilité ─────────────────────────────────────────────
# Patch 1 : NumPy 2.x       — attributs supprimés
# Patch 2 : torchvision 0.16+ — functional_tensor supprimé
# Patch 3 : PyTorch 2.x     — weights_only=True par défaut bloque les vieux .pth

import os, re, glob, subprocess, sys, sysconfig

SADTALKER_SRC = "/content/SadTalker"

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).returncode == 0

# ── Patch 1 : NumPy 2.x ───────────────────────────────────────────────────────
NUMPY_PATTERNS = [
    (r"np\.VisibleDeprecationWarning", "DeprecationWarning"),
    (r"\bnp\.bool\b(?!\w)",    "bool"),
    (r"\bnp\.int\b(?!\w)",     "int"),
    (r"\bnp\.float\b(?!\w)",   "float"),
    (r"\bnp\.complex\b(?!\w)", "complex"),
    (r"\bnp\.object\b(?!\w)",  "object"),
    (r"\bnp\.str\b(?!\w)",     "str"),
]
numpy_patched = []
for filepath in glob.glob(f"{SADTALKER_SRC}/**/*.py", recursive=True):
    try:
        src = open(filepath, "r", encoding="utf-8", errors="ignore").read()
        mod = src
        for pat, rep in NUMPY_PATTERNS:
            mod = re.sub(pat, rep, mod)
        if mod != src:
            open(filepath, "w", encoding="utf-8").write(mod)
            numpy_patched.append(os.path.relpath(filepath, SADTALKER_SRC))
    except Exception as e:
        print(f"  ⚠️  {filepath} : {e}")
print(f"✅ NumPy 2.x — {len(numpy_patched)} fichier(s) patché(s)"
      + (f" : {', '.join(numpy_patched)}" if numpy_patched else ""))

# ── Patch 2 : torchvision functional_tensor ───────────────────────────────────
OLD_TV = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
NEW_TV = "from torchvision.transforms.functional import rgb_to_grayscale"

degradations = None
try:
    import basicsr
    degradations = os.path.join(os.path.dirname(basicsr.__file__), "data", "degradations.py")
except ImportError:
    for p in sys.path:
        c = os.path.join(p, "basicsr", "data", "degradations.py")
        if os.path.isfile(c):
            degradations = c; break
if not degradations:
    r = subprocess.run("find /usr /opt /root -name 'degradations.py' -path '*/basicsr/*' 2>/dev/null | head -1",
                       shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        degradations = r.stdout.strip()

if degradations and os.path.isfile(degradations):
    content = open(degradations).read()
    if OLD_TV in content:
        sh(f"sed -i 's|{OLD_TV}|{NEW_TV}|g' \"{degradations}\"")
        print("✅ torchvision compat — degradations.py patché")
    else:
        print("✅ torchvision compat — déjà corrigé")
else:
    print("⚠️  basicsr/degradations.py introuvable — shim sitecustomize actif")

SITECUSTOMIZE = os.path.join(sysconfig.get_path("stdlib"), "sitecustomize.py")
SHIM_MARKER = "# sadtalker-torchvision-shim"
SHIM_CODE = f"""
{SHIM_MARKER}
try:
    import sys, types
    import torchvision.transforms.functional as _tvf
    _m = types.ModuleType("torchvision.transforms.functional_tensor")
    _m.rgb_to_grayscale = _tvf.rgb_to_grayscale
    sys.modules.setdefault("torchvision.transforms.functional_tensor", _m)
except Exception:
    pass
"""
existing = open(SITECUSTOMIZE).read() if os.path.isfile(SITECUSTOMIZE) else ""
if SHIM_MARKER not in existing:
    open(SITECUSTOMIZE, "a").write(SHIM_CODE)
    print("✅ sitecustomize.py — shim ajouté")
else:
    print("✅ sitecustomize.py — shim actif")

# ── Patch 3 : PyTorch 2.x — weights_only=False ────────────────────────────────
# Le regex précédent s'arrêtait à la 1ère ')' et cassait les appels imbriqués
# comme torch.load(..., map_location=torch.device(device)).
# On utilise un parser qui compte les parenthèses pour trouver la vraie fermeture.

def patch_torch_load(source):
    """Ajoute weights_only=False à chaque torch.load() en gérant les parens imbriquées."""
    MARKER = "torch.load("
    out = []
    i = 0
    while i < len(source):
        idx = source.find(MARKER, i)
        if idx == -1:
            out.append(source[i:])
            break
        out.append(source[i:idx + len(MARKER)])
        # Trouver la parenthèse fermante correspondante
        depth = 1
        j = idx + len(MARKER)
        while j < len(source) and depth > 0:
            if source[j] == '(':
                depth += 1
            elif source[j] == ')':
                depth -= 1
            if depth > 0:
                j += 1
        # source[idx+len(MARKER):j] = contenu entre les parens
        args = source[idx + len(MARKER):j]
        if "weights_only" not in args:
            out.append(args + ", weights_only=False")
        else:
            out.append(args)
        out.append(")")
        i = j + 1
    return "".join(out)

torch_patched = []
for filepath in glob.glob(f"{SADTALKER_SRC}/**/*.py", recursive=True):
    try:
        src = open(filepath, "r", encoding="utf-8", errors="ignore").read()
        if "torch.load(" not in src:
            continue
        mod = patch_torch_load(src)
        if mod != src:
            open(filepath, "w", encoding="utf-8").write(mod)
            torch_patched.append(os.path.relpath(filepath, SADTALKER_SRC))
    except Exception as e:
        print(f"  ⚠️  {filepath} : {e}")
print(f"✅ PyTorch 2.x weights_only — {len(torch_patched)} fichier(s) patché(s)"
      + (f" : {', '.join(torch_patched)}" if torch_patched else ""))

In [ ]:
# ─── Installation des dépendances Python ──────────────────────────────────────
# Colab fournit déjà PyTorch + CUDA — on ne le réinstalle pas.
# Chaque paquet est installé séparément pour isoler les erreurs.

import subprocess, sys, time

def run(cmd, label="", show_output=False):
    t = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = result.returncode == 0
    elapsed = time.time() - t
    if label:
        print(f"  {'✅' if ok else '⚠️ '} {label} ({elapsed:.0f}s)")
    if show_output and result.stdout:
        print(result.stdout[-500:])
    if not ok and result.stderr:
        last = [l for l in result.stderr.splitlines() if l.strip()]
        if last:
            print(f"     → {last[-1]}")
    return ok

print("🐍 Installation des dépendances SadTalker...\n")

# Paquets Python — dlib retiré (non utilisé par SadTalker, compile trop longtemps)
PACKAGES = [
    ("scipy",                 "scipy"),
    ("Pillow",                "Pillow"),
    ("imageio",               "imageio"),
    ("imageio-ffmpeg",        "imageio-ffmpeg"),
    ("librosa",               "librosa"),
    ("resampy",               "resampy"),
    ("pydub",                 "pydub"),
    ("yacs",                  "yacs"),
    ("tqdm",                  "tqdm"),
    ("einops",                "einops"),
    ("av",                    "av (PyAV)"),
    ("kornia",                "kornia"),
    ("basicsr",               "basicsr"),
    ("facexlib",              "facexlib"),
    ("gfpgan",                "gfpgan"),
    ("face_alignment",        "face_alignment"),
    ("safetensors",           "safetensors"),
    ("huggingface_hub",       "huggingface_hub"),
    ("batch_face",            "batch_face"),
]

failed = []
for pkg, label in PACKAGES:
    ok = run(f"pip install -q '{pkg}'", label)
    if not ok:
        failed.append(pkg)

print()
if failed:
    print(f"⚠️  {len(failed)} paquet(s) en échec (souvent non bloquants) :")
    for p in failed:
        print(f"   • {p}")
else:
    print("✅ Toutes les dépendances installées.")

import torch, numpy as np
print(f"\n📋 Environnement :")
print(f"   PyTorch : {torch.__version__} | CUDA : {torch.version.cuda}")
print(f"   NumPy   : {np.__version__}")
assert torch.cuda.is_available(), "❌ GPU non disponible — activez le GPU T4."

In [ ]:
# ─── Téléchargement des modèles pré-entraînés ─────────────────────────────────
import os, shutil
from huggingface_hub import snapshot_download, hf_hub_download

CHECKPOINT_DIR = "/content/SadTalker/checkpoints"
GFPGAN_DIR     = "/content/SadTalker/gfpgan/weights"
BFM_DIR        = f"{CHECKPOINT_DIR}/BFM_Fitting"
for d in [CHECKPOINT_DIR, GFPGAN_DIR, BFM_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Étape 1 : snapshot_download du repo vinthony/SadTalker ────────────────────
HF_CACHE = "/content/hf_sadtalker"
if not os.path.isdir(HF_CACHE) or not os.listdir(HF_CACHE):
    print("🤖 Téléchargement du repo SadTalker (10-20 min)...\n")
    try:
        snapshot_download(
            repo_id="vinthony/SadTalker",
            local_dir=HF_CACHE,
            ignore_patterns=["*.md", "*.txt", "*.py"],
        )
        print(f"✅ Repo téléchargé dans {HF_CACHE}")
    except Exception as e:
        print(f"❌ snapshot_download : {e}")
else:
    print(f"✅ Repo déjà en cache : {HF_CACHE}")

# ── Étape 2 : copie vers les répertoires SadTalker ────────────────────────────
if os.path.isdir(HF_CACHE):
    print("\n📁 Organisation des fichiers...")
    copied = 0
    for root, dirs, files in os.walk(HF_CACHE):
        for fname in files:
            if fname.endswith(".metadata") or fname in [".gitattributes", ".gitignore",
                                                         "CACHEDIR.TAG", ".DS_Store"]:
                continue
            src = os.path.join(root, fname)
            rel = os.path.relpath(root, HF_CACHE)
            dst = os.path.join(BFM_DIR if "BFM_Fitting" in rel else CHECKPOINT_DIR, fname)
            if not os.path.isfile(dst):
                shutil.copy2(src, dst)
                print(f"  ✓ {fname}")
                copied += 1
    print(f"  → {copied} fichier(s) copié(s)")

# ── Étape 3 : BFM_model_front.mat (téléchargement ciblé) ─────────────────────
# Ce fichier n'est pas inclus dans le snapshot — on le télécharge séparément.
BFM_FRONT = f"{BFM_DIR}/BFM_model_front.mat"
if not os.path.isfile(BFM_FRONT) or os.path.getsize(BFM_FRONT) < 100_000:
    print("\n📐 Téléchargement de BFM_model_front.mat...")
    try:
        hf_hub_download(
            repo_id="vinthony/SadTalker",
            filename="BFM_Fitting/BFM_model_front.mat",
            local_dir=BFM_DIR,
            local_dir_use_symlinks=False,
        )
        # Déplacer si hf_hub_download a créé un sous-dossier
        nested = f"{BFM_DIR}/BFM_Fitting/BFM_model_front.mat"
        if os.path.isfile(nested) and not os.path.isfile(BFM_FRONT):
            shutil.move(nested, BFM_FRONT)
        sz = os.path.getsize(BFM_FRONT) if os.path.isfile(BFM_FRONT) else 0
        print(f"  ✅ BFM_model_front.mat ({sz//1_000_000} MB)" if sz > 100_000
              else f"  ❌ Toujours manquant ({sz} o)")
    except Exception as e:
        print(f"  ❌ {e}")
else:
    print(f"\n✅ BFM_model_front.mat ({os.path.getsize(BFM_FRONT)//1_000_000} MB)")

# ── Étape 4 : vérification ────────────────────────────────────────────────────
# Note : SadTalker_V0.0.2_*.safetensors sont OPTIONNELS.
# SadTalker utilise automatiquement les .pth.tar si les safetensors manquent.
REQUIRED = {
    f"{CHECKPOINT_DIR}/epoch_20.pth":                    (20_000_000, "alignment model"),
    f"{CHECKPOINT_DIR}/facevid2vid_00189-model.pth.tar": (200_000_000, "face renderer"),
    f"{CHECKPOINT_DIR}/auido2exp_00300-model.pth":       (5_000_000,  "audio→expression"),
    f"{CHECKPOINT_DIR}/auido2pose_00140-model.pth":      (5_000_000,  "audio→pose"),
    f"{BFM_DIR}/BFM_model_front.mat":                    (100_000,    "3D face model"),
    f"{BFM_DIR}/01_MorphableModel.mat":                  (100_000,    "morphable model"),
    f"{BFM_DIR}/Exp_Pca.bin":                            (1_000_000,  "expression PCA"),
}
print("\n🔍 Vérification des fichiers critiques...")
missing = []
for path, (min_sz, label) in REQUIRED.items():
    sz = os.path.getsize(path) if os.path.isfile(path) else 0
    ok = sz >= min_sz
    print(f"  {'✅' if ok else '❌'} {os.path.basename(path)} — {label} ({sz//1_000_000} MB)")
    if not ok:
        missing.append(os.path.basename(path))

# ── Étape 5 : modèles GFPGAN (optionnels — nécessite ENHANCE_FACE=False si absent)
print("\n🎨 Modèles GFPGAN (optionnels)...")
try:
    from basicsr.utils.download_util import load_file_from_url
    GFPGAN_URLS = {
        "GFPGANv1.4.pth":
            "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
        "alignment_WFLW_4HG.pth":
            "https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth",
        "detection_Resnet50_Final.pth":
            "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",
        "parsing_parsenet.pth":
            "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth",
    }
    for fname, url in GFPGAN_URLS.items():
        dest = f"{GFPGAN_DIR}/{fname}"
        if os.path.isfile(dest) and os.path.getsize(dest) > 1_000_000:
            print(f"  ✓ {fname} ({os.path.getsize(dest)//1_000_000} MB)")
            continue
        print(f"  ⬇️  {fname}...")
        try:
            load_file_from_url(url, model_dir=GFPGAN_DIR, progress=True, file_name=fname)
            sz = os.path.getsize(dest) if os.path.isfile(dest) else 0
            print(f"  {'✅' if sz > 1_000_000 else '❌'} {fname} ({sz//1_000_000} MB)")
        except Exception as e:
            print(f"  ⚠️  {fname} : {e}")
except ImportError:
    print("  ⚠️  basicsr absent → GFPGAN non disponible")
    print("  💡 Passez ENHANCE_FACE = False à l'étape 5 pour générer sans amélioration du visage")

print()
if missing:
    print(f"❌ Fichiers critiques manquants : {missing}")
else:
    print("✅ Tous les modèles critiques sont présents !")

---
## 🖼️ Étape 3 — Upload de l'image source

In [ ]:
# ─── Upload de l'image PNG ────────────────────────────────────────────────────
# Conseils pour une meilleure qualité :
#   • Image carrée recommandée (ex: 512×512 ou 1024×1024)
#   • Visage bien centré, bien éclairé, de face ou légèrement de profil
#   • Fond uni de préférence pour un meilleur résultat
#   • Formats acceptés : PNG, JPG

import os, shutil
from google.colab import files
from IPython.display import display, Image as IPImage
import ipywidgets as widgets

IMAGE_DIR = "/content/inputs"
os.makedirs(IMAGE_DIR, exist_ok=True)

print("🖼️  Choisissez votre image source (PNG/JPG) :")
print("   → Visage bien visible, centré, fond uni de préférence")
print()

uploaded_img = files.upload()

if not uploaded_img:
    raise FileNotFoundError("❌ Aucun fichier uploadé. Relancez cette cellule.")

# Récupération et validation du fichier
img_filename = list(uploaded_img.keys())[0]
img_ext      = os.path.splitext(img_filename)[1].lower()

if img_ext not in [".png", ".jpg", ".jpeg"]:
    raise ValueError(f"❌ Format non supporté : {img_ext}. Utilisez PNG ou JPG.")

SOURCE_IMAGE = f"{IMAGE_DIR}/source_image{img_ext}"
shutil.move(img_filename, SOURCE_IMAGE)

print(f"✅ Image chargée : {img_filename}")

# Affichage de l'image uploadée
from PIL import Image as PILImage
img = PILImage.open(SOURCE_IMAGE)
print(f"   Dimensions : {img.width} × {img.height} px | Mode : {img.mode}")

# Avertissement si l'image est trop petite
if img.width < 256 or img.height < 256:
    print("⚠️  Image petite (< 256px) — la qualité peut être réduite.")

display(IPImage(SOURCE_IMAGE, width=300))

---
## 🎵 Étape 4 — Upload de l'audio

In [ ]:
# ─── Upload du fichier audio ──────────────────────────────────────────────────
# Conseils pour un meilleur lip sync :
#   • Voix claire, sans bruit de fond si possible
#   • Durée conseillée : 5 à 30 secondes (au-delà la génération est plus longue)
#   • Formats acceptés : MP3, WAV, M4A
#   • Le fichier sera automatiquement converti en WAV 16kHz mono

import os, shutil
from google.colab import files
from IPython.display import Audio, display

AUDIO_DIR = "/content/inputs"
os.makedirs(AUDIO_DIR, exist_ok=True)

print("🎵 Choisissez votre fichier audio (MP3 / WAV / M4A) :")
print("   → Voix claire, durée recommandée : 5-30 secondes")
print()

uploaded_audio = files.upload()

if not uploaded_audio:
    raise FileNotFoundError("❌ Aucun fichier uploadé. Relancez cette cellule.")

# Récupération et validation du fichier
audio_filename = list(uploaded_audio.keys())[0]
audio_ext      = os.path.splitext(audio_filename)[1].lower()

if audio_ext not in [".mp3", ".wav", ".m4a", ".ogg", ".flac"]:
    raise ValueError(f"❌ Format non supporté : {audio_ext}. Utilisez MP3, WAV ou M4A.")

raw_audio = f"{AUDIO_DIR}/audio_raw{audio_ext}"
shutil.move(audio_filename, raw_audio)

# Conversion en WAV 16kHz mono (format requis par SadTalker)
DRIVEN_AUDIO = f"{AUDIO_DIR}/driven_audio.wav"
print("🔄 Conversion en WAV 16kHz mono...")
ok = run(f'ffmpeg -y -i "{raw_audio}" -ar 16000 -ac 1 -f wav "{DRIVEN_AUDIO}" -loglevel error')

if ok:
    # Calcul de la durée
    dur_raw = subprocess.check_output(
        f'ffprobe -v error -show_entries format=duration -of csv=p=0 "{DRIVEN_AUDIO}"',
        shell=True
    ).decode().strip()
    duration = float(dur_raw) if dur_raw else 0
    print(f"✅ Audio prêt : {audio_filename} ({duration:.1f} s)")
    if duration > 60:
        print("⚠️  Audio long (> 60s) : la génération peut prendre plusieurs minutes.")
else:
    raise RuntimeError("❌ Conversion audio échouée.")

# Lecture de prévisualisation
display(Audio(DRIVEN_AUDIO))

---
## ⚙️ Étape 5 — Paramètres de génération

In [ ]:
# ─── Paramètres de génération ─────────────────────────────────────────────────

# "256" → rapide (~2 min) | "512" → meilleure qualité (~5 min)
SIZE = "512"

# GFPGAN améliore la qualité mais nécessite des modèles supplémentaires.
# Mettez True uniquement si les modèles GFPGAN sont bien téléchargés.
# Si l'étape 2 a affiché "basicsr absent", laissez False.
ENHANCE_FACE = False

# "crop" → recadre sur le visage (meilleur lip sync)
# "resize" → redimensionne toute l'image | "full" → corps entier
PREPROCESS = "crop"

# Amélioration du fond (nécessite GFPGAN)
BG_ENHANCER = False

OUTPUT_DIR = "/content/outputs"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("⚙️  Configuration de génération :")
print(f"   • Résolution        : {SIZE}px")
print(f"   • Amélioration face : {'✅ GFPGAN activé' if ENHANCE_FACE else '❌ désactivé (mode rapide)'}")
print(f"   • Prétraitement     : {PREPROCESS}")
print(f"   • Sortie            : {OUTPUT_DIR}")
if not ENHANCE_FACE:
    print()
    print("   💡 Pour activer GFPGAN : relancez l'étape 1 et vérifiez que")
    print("      les modèles GFPGAN sont bien téléchargés, puis mettez ENHANCE_FACE=True")

---
## 🚀 Étape 6 — Génération de la vidéo

In [ ]:
# ─── Génération de la vidéo avec SadTalker ───────────────────────────────────
# SadTalker fonctionne en 3 phases internes :
#   1. Extraction des landmarks du visage source
#   2. Conversion audio → coefficients de mouvement 3D
#   3. Rendu vidéo final avec le modèle de génération de visage

import os, glob, time, subprocess
from IPython.display import HTML, display

os.chdir("/content/SadTalker")

# Construction de la commande d'inférence
enhancer_flag = "--enhancer gfpgan" if ENHANCE_FACE else ""
bg_flag       = "--background_enhancer" if BG_ENHANCER else ""

CMD = (
    f"python inference.py "
    f"--driven_audio '{DRIVEN_AUDIO}' "
    f"--source_image '{SOURCE_IMAGE}' "
    f"--result_dir '{OUTPUT_DIR}' "
    f"--size {SIZE} "
    f"--preprocess {PREPROCESS} "
    f"--still "
    f"{enhancer_flag} "
    f"{bg_flag}"
)

print("🚀 Lancement de SadTalker...")
print(f"   Résolution : {SIZE}px | Prétraitement : {PREPROCESS}")
print(f"   GFPGAN     : {'activé' if ENHANCE_FACE else 'désactivé'}")
print()
print("⏳ Génération en cours (2-8 min selon les paramètres)...")
print("─" * 60)

t_start = time.time()

# Exécution avec affichage progressif
process = subprocess.Popen(
    CMD, shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, cwd="/content/SadTalker"
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

elapsed = time.time() - t_start

# Récupération du fichier de sortie
mp4_files = sorted(
    glob.glob(f"{OUTPUT_DIR}/**/*.mp4", recursive=True),
    key=os.path.getmtime
)

if not mp4_files or process.returncode != 0:
    print("\n❌ Génération échouée. Vérifiez les messages d'erreur ci-dessus.")
else:
    RAW_VIDEO = mp4_files[-1]
    print(f"\n✅ Vidéo générée en {elapsed:.0f}s : {RAW_VIDEO}")

    # Aperçu de la vidéo brute (format 1:1)
    display(HTML(f"""
    <video width="400" controls autoplay loop muted>
      <source src="{RAW_VIDEO}" type="video/mp4">
    </video>
    <p><em>Aperçu — format carré (avant conversion 9:16)</em></p>
    """))

---
## 📱 Étape 7 — Export vertical 9:16 pour TikTok / Reels / Shorts

In [ ]:
# ─── Conversion en format vertical 9:16 ──────────────────────────────────────
# Le format 9:16 est le standard des vidéos courtes sur :
#   TikTok, Instagram Reels, YouTube Shorts, Snapchat Spotlight
#
# Résolutions cibles :
#   • 1080×1920 px — Full HD vertical (publication)
#   • 720×1280 px  — HD vertical (plus léger, prévisualisation)
#
# Stratégie ffmpeg :
#   1. On redimensionne la vidéo pour que la largeur = TARGET_W
#   2. On centre verticalement et on ajoute un fond flouté pour remplir 9:16
#   3. On encode en H.264 haute qualité (CRF 18)

import os, subprocess
from IPython.display import HTML, display
from google.colab import files

# ┌─ Résolution de sortie ──────────────────────────────────────────────────────
# Choisissez : "1080x1920" (Full HD, ~15 Mo) ou "720x1280" (HD, ~8 Mo)
RESOLUTION = "1080x1920"

TARGET_W, TARGET_H = map(int, RESOLUTION.split("x"))

# ┌─ Couleur du fond ───────────────────────────────────────────────────────────
# "blur"   → fond flouté à partir de la vidéo elle-même (aspect cinématique)
# "black"  → fond noir uni
# "white"  → fond blanc uni
# "#RRGGBB" → couleur personnalisée (ex: "#1a1a2e")
BACKGROUND = "blur"

EXPORT_PATH = f"{OUTPUT_DIR}/sadtalker_916_{TARGET_W}x{TARGET_H}.mp4"

print(f"📱 Export en {TARGET_W}×{TARGET_H} (9:16)...")
print(f"   Fond : {BACKGROUND}")
print()

if BACKGROUND == "blur":
    # Fond flouté : la vidéo est mise à l'échelle 9:16, puis la version originale
    # est centrée par-dessus. Résultat très propre pour les réseaux sociaux.
    FFMPEG_CMD = (
        f'ffmpeg -y -i "{RAW_VIDEO}" '
        f'-vf "'
        f'split[v1][v2];'
        f'[v1]scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase,'
        f'crop={TARGET_W}:{TARGET_H},'
        f'boxblur=20:5[bg];'
        f'[v2]scale=iw*min({TARGET_W}/iw\,{TARGET_H}/ih):ih*min({TARGET_W}/iw\,{TARGET_H}/ih),'
        f'pad={TARGET_W}:{TARGET_H}:(ow-iw)/2:(oh-ih)/2:color=00000000[fg];'
        f'[bg][fg]overlay=0:0:format=auto,'
        f'format=yuv420p'
        f'" '
        f'-c:v libx264 -crf 18 -preset slow '
        f'-c:a aac -b:a 192k '
        f'"{EXPORT_PATH}" '
        f'-loglevel warning'
    )
else:
    # Fond couleur unie : plus simple, convient aux mascottes sur fond uni
    bg_color = "black" if BACKGROUND == "black" else (
                "white" if BACKGROUND == "white" else BACKGROUND.lstrip("#")
               )
    FFMPEG_CMD = (
        f'ffmpeg -y -i "{RAW_VIDEO}" '
        f'-vf "'
        f'scale=iw*min({TARGET_W}/iw\,{TARGET_H}/ih):ih*min({TARGET_W}/iw\,{TARGET_H}/ih),'
        f'pad={TARGET_W}:{TARGET_H}:(ow-iw)/2:(oh-ih)/2:color={bg_color},'
        f'format=yuv420p'
        f'" '
        f'-c:v libx264 -crf 18 -preset slow '
        f'-c:a aac -b:a 192k '
        f'"{EXPORT_PATH}" '
        f'-loglevel warning'
    )

result = subprocess.run(FFMPEG_CMD, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ Erreur ffmpeg :")
    print(result.stderr[-2000:])
else:
    size_mb = os.path.getsize(EXPORT_PATH) / 1e6
    print(f"✅ Export réussi !")
    print(f"   Fichier : {EXPORT_PATH}")
    print(f"   Taille  : {size_mb:.1f} MB")
    print(f"   Format  : {TARGET_W}×{TARGET_H} px, H.264, AAC 192kbps")

    # Aperçu de la vidéo 9:16
    display(HTML(f"""
    <video width="280" controls autoplay loop muted
           style="border-radius:16px; box-shadow:0 4px 20px rgba(0,0,0,0.4);">
      <source src="{EXPORT_PATH}" type="video/mp4">
    </video>
    <p><em>Aperçu — Format vertical {TARGET_W}×{TARGET_H} (9:16)</em></p>
    """))

In [ ]:
# ─── Export de variantes supplémentaires (optionnel) ─────────────────────────
# Cette cellule génère plusieurs formats en une seule passe :
#   • 9:16 Full HD  (1080×1920) — publication TikTok / Reels
#   • 9:16 HD       (720×1280)  — prévisualisation, partage rapide
#   • 1:1 carré     (1080×1080) — Instagram post, Twitter
# Le fichier original 1:1 de SadTalker est conservé tel quel.

VARIANTS = [
    {"name": "TikTok_FullHD",  "w": 1080, "h": 1920, "bg": "blur"},
    {"name": "Reels_HD",       "w": 720,  "h": 1280, "bg": "blur"},
    {"name": "Carre_Instagram","w": 1080, "h": 1080, "bg": "blur"},
]

def ffmpeg_resize(src, dest, w, h, bg="blur"):
    """Redimensionne et formate une vidéo avec fond flouté ou coloré."""
    if bg == "blur":
        vf = (
            f"split[v1][v2];"
            f"[v1]scale={w}:{h}:force_original_aspect_ratio=increase,"
            f"crop={w}:{h},boxblur=20:5[bg];"
            f"[v2]scale=iw*min({w}/iw\\,{h}/ih):ih*min({w}/iw\\,{h}/ih),"
            f"pad={w}:{h}:(ow-iw)/2:(oh-ih)/2:color=00000000[fg];"
            f"[bg][fg]overlay=0:0:format=auto,format=yuv420p"
        )
    else:
        vf = (
            f"scale=iw*min({w}/iw\\,{h}/ih):ih*min({w}/iw\\,{h}/ih),"
            f"pad={w}:{h}:(ow-iw)/2:(oh-ih)/2:color={bg},format=yuv420p"
        )
    cmd = (
        f'ffmpeg -y -i "{src}" -vf "{vf}" '
        f'-c:v libx264 -crf 18 -preset slow '
        f'-c:a aac -b:a 192k "{dest}" -loglevel error'
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode == 0

print("🎬 Génération des variantes d'export...\n")
exported = []

for v in VARIANTS:
    dest = f"{OUTPUT_DIR}/sadtalker_{v['name']}_{v['w']}x{v['h']}.mp4"
    print(f"  ⏳ {v['name']} ({v['w']}×{v['h']})...", end=" ")
    ok = ffmpeg_resize(RAW_VIDEO, dest, v["w"], v["h"], v["bg"])
    if ok:
        size_mb = os.path.getsize(dest) / 1e6
        print(f"✅ {size_mb:.1f} MB")
        exported.append(dest)
    else:
        print("❌ Échec")

print(f"\n✅ {len(exported)}/{len(VARIANTS)} variantes exportées dans {OUTPUT_DIR}")

---
## 💾 Étape 8 — Téléchargement des fichiers

In [ ]:
# ─── Téléchargement des vidéos générées ──────────────────────────────────────
# Télécharge tous les fichiers MP4 présents dans le répertoire de sortie.
# Si vous ne souhaitez télécharger qu'un seul fichier, commentez les autres.

import glob, os
from google.colab import files

mp4_outputs = sorted(glob.glob(f"{OUTPUT_DIR}/*.mp4"), key=os.path.getmtime)

if not mp4_outputs:
    print("❌ Aucun fichier MP4 trouvé. Vérifiez que la génération a bien réussi.")
else:
    print(f"📥 {len(mp4_outputs)} fichier(s) prêt(s) au téléchargement :\n")
    for f_path in mp4_outputs:
        size_mb = os.path.getsize(f_path) / 1e6
        print(f"   • {os.path.basename(f_path)} — {size_mb:.1f} MB")
    print()
    print("⬇️  Téléchargement en cours...")
    for f_path in mp4_outputs:
        files.download(f_path)
    print("✅ Téléchargement lancé pour tous les fichiers.")

In [ ]:
# ─── Téléchargement sélectif (optionnel) ─────────────────────────────────────
# Utilisez cette cellule pour télécharger uniquement un fichier spécifique.
# Remplacez le nom du fichier par celui souhaité.

from google.colab import files

# Modifiez ce chemin si nécessaire :
FICHIER_A_TELECHARGER = EXPORT_PATH  # fichier 9:16 principal

if os.path.isfile(FICHIER_A_TELECHARGER):
    print(f"⬇️  Téléchargement de : {os.path.basename(FICHIER_A_TELECHARGER)}")
    files.download(FICHIER_A_TELECHARGER)
else:
    print(f"❌ Fichier introuvable : {FICHIER_A_TELECHARGER}")

---
## 🔁 Bonus — Générer une nouvelle vidéo sans tout réinstaller

In [ ]:
# ─── Régénération rapide ──────────────────────────────────────────────────────
# Si la session Colab est toujours active (environnement non réinitialisé),
# vous pouvez relancer une génération avec de nouveaux fichiers sans réinstaller.
#
# Relancez simplement les cellules dans l'ordre :
#   Étape 3 → nouvelle image
#   Étape 4 → nouveau fichier audio
#   Étape 5 → modifier les paramètres si besoin
#   Étape 6 → génération
#   Étape 7 → export 9:16
#   Étape 8 → téléchargement

print("💡 Instructions pour régénérer :")
print()
print("  1. Allez à l'Étape 3 → uploadez une nouvelle image")
print("  2. Allez à l'Étape 4 → uploadez un nouvel audio")
print("  3. Relancez les cellules Étapes 5, 6, 7, 8")
print()
print("⚠️  Si la session a été réinitialisée (runtime expired),")
print("    il faut relancer depuis l'Étape 1 (les modèles sont perdus).")
print()
print(f"📁 Fichiers actuellement dans {OUTPUT_DIR} :")
for f_path in sorted(glob.glob(f"{OUTPUT_DIR}/**", recursive=True)):
    if os.path.isfile(f_path):
        size_mb = os.path.getsize(f_path) / 1e6
        print(f"   • {os.path.relpath(f_path, OUTPUT_DIR)} — {size_mb:.1f} MB")

---
## 💡 Conseils & Astuces

### Pour un meilleur lip sync
- Utilisez une image avec le visage **bien centré** et **bien éclairé**
- Préférez une image **carrée** (512×512 ou 1024×1024)
- L'audio doit être **clair**, sans musique de fond
- La **résolution 512** donne de meilleurs résultats que 256

### Pour les réseaux sociaux
| Plateforme | Format | Résolution | Durée max |
|---|---|---|---|
| TikTok | 9:16 | 1080×1920 | 60 min |
| Instagram Reels | 9:16 | 1080×1920 | 90 s |
| YouTube Shorts | 9:16 | 1080×1920 | 60 s |
| Instagram Post | 1:1 | 1080×1080 | 60 s |
| Twitter/X | 16:9 ou 1:1 | 1280×720 | 2 min 20 s |

### Résolution des problèmes courants
- **`np.VisibleDeprecationWarning` / AttributeError numpy** → La cellule de patch (Étape 2) corrige automatiquement ce bug NumPy 2.x. Si l'erreur persiste, relancez depuis l'Étape 1.
- **CUDA out of memory** → Réduisez `SIZE` à `"256"` et désactivez `ENHANCE_FACE`
- **No face detected** → Utilisez une image avec un visage plus grand et plus net
- **Session expirée** → Relancez depuis l'Étape 1 (les modèles sont à re-télécharger)
- **Vidéo floue** → Activez `ENHANCE_FACE = True` et utilisez `SIZE = "512"`

---
*Notebook créé pour le projet David-GERBER.fr — Propulsé par [SadTalker](https://github.com/OpenTalker/SadTalker)*